# Kangaroo Island (Australia) Bushfires (2019-2020)

## Notebook 1: Data Preparation

This Notebook:
1. Locates the Kangaroo Island to extract satellite Imagery
2. Utilises Google Earth Engine and Sentinel-2 Satellite data to extract maps
3. Extracts Pre-fire and Post Fire Images for further analysis
4. Summarises Environmental Impact of operating the notebook

## Important:
To run this notebook, you will need a Google account as well as allow permission for this notebook to access your Google drive and Google Earth Engine. You can use Google Earth Engine for free with a non-commercial academic account.


## Step 1: Connecting Notebook with Google Drive

In [ ]:
from google.colab import drive
import os

# Connecting notebook with Google Drive
drive.mount('/content/drive')

# setting up project path
project_path = '/content/drive/MyDrive/AI4EO/AI4EO_Main_Project'


## Step 2: Setup the Emissions Tracker to assess Environmental Impact

In [ ]:
# Install tools
!pip install -q codecarbon geemap

# Load the tools into the notebook's memory
import ee
import geemap
from codecarbon import EmissionsTracker

# setup emissions tracker
tracker = EmissionsTracker(output_dir=project_path)
tracker.start()

print("Tools are loaded and carbon tracking has started!")

## Step 3: Locate Kangaroo Island and Generate Map

Important: This section will not run properly if you do not have a Google Earth Engine account with a valid project id

In [ ]:
import ee
import geemap

# connect with Earth Engine
ee.Authenticate()

#  Initialize using your project ID
ee.Initialize(project='ai4eobushfireproject')

#  locate kangaroo island
# [Lon_Left, Lat_Bottom, Lon_Right, Lat_Top]
roi = ee.Geometry.Rectangle([136.5, -36.1, 138.1, -35.6])

# Verify whether satellite image is connected
test_img = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED").first()


# create a map of kangaroo island
Map = geemap.Map()
Map.centerObject(roi, 9)
Map.addLayer(roi, {'color': 'red'}, 'Study Area')
Map

## Step 4: Extract Pre-Fire Image

In [ ]:
# sentinel-2 Dec: 2019 data
pre_fire_col = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
    .filterBounds(roi)
    .filterDate('2019-12-01', '2019-12-31')
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 10)))

# create cloud free image which is a median of images from Dec-2019
pre_fire_img = pre_fire_col.median().clip(roi)

# setup 4 bands
viz_params = {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 3000}

# 4 bands on map
Map.addLayer(pre_fire_img, viz_params, 'Pre-Fire (Healthy Forest)')

## Step 5: Extract Post-Fire Image

In [ ]:
# extract Feb-2020 images
post_fire_col = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
    .filterBounds(roi)
    .filterDate('2020-02-01', '2020-02-28')
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 10)))

# median image of sentinel-2 data from Feb-2020
post_fire_img = post_fire_col.median().clip(roi)

# add on map
Map.addLayer(post_fire_img, viz_params, 'Post-Fire (Burned)')


## Step 6: Save Images to folder on Google Drive

Important: Move files manually to a folder where you want them to be stored

In [ ]:
# save pre-fire image
task_pre = ee.batch.Export.image.toDrive(
    image=pre_fire_img.select(['B4', 'B3', 'B2', 'B8']),
    description='Kangaroo_Island_Pre_Fire',
    folder='AI4EO/AI4EO_Main_Project/data',
    region=roi,
    scale=20,
    fileFormat='GeoTIFF'
)
task_pre.start()

# save post-fire image
task_post = ee.batch.Export.image.toDrive(
    image=post_fire_img.select(['B4', 'B3', 'B2', 'B8']),
    description='Kangaroo_Island_Post_Fire',
    folder='AI4EO/AI4EO_Main_Project/data',
    region=roi,
    scale=20,
    fileFormat='GeoTIFF'
)
task_post.start()


## Assess Environmental Impact for Notebook 1

In [ ]:
# terminate emissions tracker
emissions_data = tracker.stop()

print(f"Tracking complete! Your carbon report is saved in {project_path}")
print(f"Estimated CO2 emissions for this session: {emissions_data:.6f} kg")